In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt

In [ ]:
df = pd.read_csv("spam.csv", encoding="latin1")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df = df.drop(columns=["Unnamed: 2","Unnamed: 3","Unnamed: 4"])

In [ ]:
df["v1"].value_counts()

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['v1'] = le.fit_transform(df['v1'])

In [ ]:
x = df["v2"]
y = df["v1"]

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tok = Tokenizer(num_words=5000, oov_token="OOV")
tok.fit_on_texts(x_train)

x_train_seq = tok.texts_to_sequences(x_train)
x_test_seq = tok.texts_to_sequences(x_test)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

x_train_pad = pad_sequences(x_train_seq,maxlen=50, padding="post", truncating="post")
x_test_pad = pad_sequences(x_test_seq,maxlen=50, padding="post", truncating="post")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Dense,Embedding, SimpleRNN

model = Sequential([
    Input(shape=(50,)),
    Embedding(5000, 32),
    SimpleRNN(32,activation="tanh"),
    Dense(1,activation="sigmoid")

])

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    x_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

In [ ]:
loss , accuracy = model.evaluate(x_test_pad,y_test)
print(loss)
print(accuracy)

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

During backpropagation through time (BPTT), gradients are repeatedly multiplied across time steps and can shrink toward zero. This makes it difficult for a SimpleRNN to learn long-range dependencies because earlier time steps receive very small gradient updates.